# Classification binaire de texte — Jeu de données IMDB

Réseau neuronal à propagation avant (couches **Dense**) pour classer les critiques de films en **positives (1)** ou **négatives (0)**.

Ce notebook :
1. prétraite les données textuelles (encodage one-hot)
2. construit un modèle dense (2 couches cachées ReLU + sortie sigmoïde)
3. entraîne sur 20 époques avec suivi de validation
4. visualise perte et précision pour détecter le surapprentissage
5. ré-entraîne avec un nombre optimal d'époques et évalue sur le test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import models, layers

## 1. Prétraitement des données

In [ ]:
num_words = 10_000  # on garde les 10 000 tokens les plus fréquents

# Chargement du jeu de données IMDB depuis Keras
(train_data, train_labels), (test_data, test_labels) = \
    keras.datasets.imdb.load_data(num_words=num_words)

print("Nombre de critiques d'entraînement :", len(train_data))
print("Nombre de critiques de test        :", len(test_data))
print("Exemple (début de la 1re critique) :", train_data[0][:10])

In [ ]:
# Les données sont des listes d'entiers -> encodage one-hot en vecteurs 10000-D
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0  # met à 1 les indices présents dans la critique
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# Les étiquettes sont des scalaires (0/1) -> conversion en float32
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print("Forme de x_train :", x_train.shape)

In [ ]:
# Découpage entraînement / validation
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

## 2. Construction du modèle

In [ ]:
model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(num_words,)),  # 1re couche cachée
    layers.Dense(16, activation="relu"),                            # 2e couche cachée
    layers.Dense(1, activation="sigmoid"),                          # sortie binaire
])

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 3. Entraînement du modèle (20 époques, batch de 512)

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

## 4. Évaluation : courbes de perte et de précision

In [ ]:
history_dict = history.history
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
epochs = range(1, len(loss) + 1)

# Courbe de perte
plt.figure()
plt.plot(epochs, loss, "bo", label="Perte entraînement")
plt.plot(epochs, val_loss, "b", label="Perte validation")
plt.title("Perte entraînement et validation")
plt.xlabel("Époques")
plt.ylabel("Perte")
plt.legend()
plt.show()

In [ ]:
# Courbe de précision
plt.figure()
plt.plot(epochs, acc, "bo", label="Précision entraînement")
plt.plot(epochs, val_acc, "b", label="Précision validation")
plt.title("Précision entraînement et validation")
plt.xlabel("Époques")
plt.ylabel("Précision")
plt.legend()
plt.show()

On observe que la perte de validation cesse de diminuer et repart à la hausse après quelques époques (~4) : c'est le **surapprentissage**. On ré-entraîne donc un modèle neuf sur ce nombre optimal d'époques.

In [ ]:
final_model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(num_words,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
final_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
final_model.fit(x_train, y_train, epochs=4, batch_size=512)

## 5. Analyse des résultats : évaluation sur l'ensemble de test

In [ ]:
test_loss, test_acc = final_model.evaluate(x_test, y_test)
print(f"Perte sur l'ensemble de test     : {test_loss:.4f}")
print(f"Précision sur l'ensemble de test : {test_acc:.4f}")